# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

causal-self-attention就是GPT里面的，每个注意力只能看到之前的，不能看到之后的

在实际计算的过程中，我们把QK矩阵弄成下三角的就行了

流程是，首先生成一个mask的矩阵，他的右上角都是true

In [3]:
import torch
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    #这里相当于是给我们弄好的QKV了，首先是正常的去计算这个score就行
    d_k = K.size(-1)
    score = Q@K.transpose(1,2)/math.sqrt(d_k)

    #然后我们准备一个mask矩阵，这个矩阵的右上角弄成True
    #首先先拿到和score一样的维度
    S = score.size(-1)
    mask = torch.triu(torch.ones(S,S,device = score.device,dtype = bool,),diagonal = 1 )
    #解释一下，torch.triu是生成一个上三角矩阵，矩阵中处于下三角和主对角线上的元素变成 False，其他弄成True，
    # 这里的diagonal是=1，意思是对角线上可以自己看自己，所以保留对角线上是True
    #mask是一个S*S的方阵,下面的squeeze0意思是把它的0维度加一个，相当于弄成1 S S，这样可以和scroe保持一样
    score = score.masked_fill(mask.squeeze(0),float('-inf'))

    #最后过softmax
    return torch.softmax(score,dim = -1)@V

      # Replace this

In [7]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [8]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.8ms)
  ✅ [2/4] Future tokens don't affect past (2.4ms)
  ✅ [3/4] First position only sees itself (0.5ms)
  ✅ [4/4] Gradient flow (15.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (19.9ms total)
  Progress saved. Run status() to see your dashboard.

